In [6]:
import numpy as np 
from time import time
import pennylane as qml
import pennylane.estimator as qre

# Estimating workflows using existing PannyLane workflows

In [24]:
n_cells = [25, 25]
kx, ky, kz = (0.5, 0.6, 0.7)

In [25]:
t1 = time()
flat_hamiltonian = qml.spin.kitaev(n_cells, coupling=np.array([kx, ky, kz]))
flat_hamiltonian.compute_grouping()

In [26]:
groups = []
for group_indices in flat_hamiltonian.grouping_indices:
    grouped_term = qml.sum(*(flat_hamiltonian.operands[index] for index in group_indices))
    groups.append(grouped_term)


In [27]:
grouped_hamiltonian = qml.sum(*groups)
t2 = time()
t_generation = t2 - t1

In [28]:
num_steps = 10
order = 6

@qml.qnode(qml.device('ionq.qpu', backend="harmony", wires=2))
def executable_circuit(hamiltonian, num_steps, order):
    for wire in hamiltonian.wires:
        qml.Hadamard(wire)
    qml.TrotterProduct(hamiltonian, time=1.0, n=num_steps, order=order)
    return qml.state()

In [29]:
t1 =  time()
resources_exec = qre.estimate(executable_circuit)(grouped_hamiltonian, num_steps, order)
t2 = time()

print(f"Processing time: {t2 - t1:.2f} seconds")
print(resources_exec)

Processing time: 2.17 seconds
--- Resources: ---
 Total wires: 1250
   algorithmic wires: 1250
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 2.972E+7
   'T': 2.670E+7,
   'CNOT': 1.214E+6,
   'Z': 6.000E+5,
   'S': 1.200E+6,
   'Hadamard': 1.250E+3


# Fast estimation

In [30]:
n_cell = 100

def pauli_quantities(n_cell):
    n_q = 2 * n_cell**2
    n_xx = n_cell**2
    n_yy = n_cell * (n_cell - 1)
    n_zz = n_yy
    return n_q, n_xx, n_yy, n_zz

n_q, n_xx, n_yy, n_zz = pauli_quantities(n_cell)

In [32]:
pauli_word_distribution = {"XX": n_xx, "YY": n_yy, "ZZ": n_zz}

kitaev_H = qre.PauliHamiltonian(
    num_qubits=n_q,
    pauli_terms=pauli_word_distribution,
)

In [34]:
def circuit(hamiltonian, num_steps, order):
    qre.UniformStatePrep(num_states=2 ** n_q)
    qre.TrotterPauli(hamiltonian, num_steps, order)

In [35]:
t1 = time()
resources_exec = qre.estimate(circuit)(kitaev_H, num_steps, order)
t2 = time()

print(f"Processing time: {t2 - t1:.2f} seconds")
print(resources_exec)

Processing time: 0.00 seconds
--- Resources: ---
 Total wires: 2.000E+4
   algorithmic wires: 20000
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 7.151E+8
   'T': 6.556E+8,
   'CNOT': 2.980E+7,
   'Z': 9.900E+6,
   'S': 1.980E+7,
   'Hadamard': 2.000E+4


# Single component resource

In [36]:
resources_without_grouping = qre.estimate(qre.TrotterPauli(kitaev_H, num_steps, order))

In [37]:
commuting_groups = [{"XX": n_xx}, {"YY": n_yy}, {"ZZ": n_zz}]

kitaev_H_with_grouping = qre.PauliHamiltonian(
    num_qubits=n_q,
    pauli_terms=commuting_groups,
)

resources_with_grouping = qre.estimate(
    qre.TrotterPauli(kitaev_H_with_grouping, num_steps, order)
)

In [38]:
# Just compare T gates:
t_count_1 = resources_without_grouping.gate_counts["T"]
t_count_2 = resources_with_grouping.gate_counts["T"]
reduction = abs((t_count_2 - t_count_1) / t_count_1)
print("--- Without grouping ---", f"\n T gate count: {t_count_1:.3E}\n")
print("--- With grouping ---", f"\n T gate count: {t_count_2:.3E}\n")
print(f"Difference: {100*reduction:.1f}% reduction")

--- Without grouping --- 
 T gate count: 6.556E+08

--- With grouping --- 
 T gate count: 4.371E+08

Difference: 33.3% reduction


# Changeing gates

In [41]:
res = qre.estimate(circuit)(kitaev_H_with_grouping, num_steps, order)
print(res)

--- Resources: ---
 Total wires: 2.000E+4
   algorithmic wires: 20000
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 4.867E+8
   'T': 4.371E+8,
   'CNOT': 1.987E+7,
   'Z': 9.900E+6,
   'S': 1.980E+7,
   'Hadamard': 2.000E+4


In [42]:
highlvl_gateset = {
    "RX","RY","RZ",
    "Toffoli",
    "X","Y","Z",
    "Adjoint(S)","Adjoint(T)",
    "Hadamard","S","CNOT","T",
}

highlvl_res = qre.estimate(
    circuit,
    gate_set=highlvl_gateset,
)(kitaev_H_with_grouping, num_steps, order)

print(f"High-level resources:\n{highlvl_res}\n")

High-level resources:
--- Resources: ---
 Total wires: 2.000E+4
   algorithmic wires: 20000
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 4.962E+7
   'RX': 2.510E+6,
   'RY': 4.950E+6,
   'Adjoint(S)': 9.900E+6,
   'RZ': 2.475E+6,
   'CNOT': 1.987E+7,
   'S': 9.900E+6,
   'Hadamard': 2.000E+4

